# 04_inference_demo.ipynb — Inference & Alerting Demo

> **Project:** SentinelFatal2 — Foundation Model for fetal distress prediction from CTG  
> **Source:** arXiv:2601.06149v1, Section II-F, Figure 5  
> **SSOT:** `docs/work_plan.md`, Part ו, Step 5  
> **Agent:** Agent 5

This notebook demonstrates the full Stage 1 + Stage 2 alerting pipeline on a single recording:
1. Load fine-tuned model
2. Run sliding-window inference → continuous score timeline
3. Extract alert segments (score > 0.5)
4. Compute 4 alert features
5. Visualize score timeline with alert regions

**Validations:** V5.1 – V5.6

---
**NOTE:** LR training is done via `src/train/train_lr.py`, not this notebook.
This notebook is demonstration only.

In [ ]:
# Reproducibility seeds (run first in every notebook)
import torch
import random
import numpy as np

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
print(f'[seed] All seeds set to {SEED}')

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# --- Project root ---
NOTEBOOK_DIR = Path(os.path.abspath(''))
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

from src.model.patchtst import PatchTST, load_config
from src.model.heads import ClassificationHead
from src.inference.sliding_window import (
    INFERENCE_STRIDE_REPRO,
    INFERENCE_STRIDE_RUNTIME,
    inference_recording,
)
from src.inference.alert_extractor import (
    ALERT_THRESHOLD,
    extract_alert_segments,
    compute_alert_features,
    ZERO_FEATURES,
)

print(f'INFERENCE_STRIDE_REPRO   = {INFERENCE_STRIDE_REPRO}')
print(f'INFERENCE_STRIDE_RUNTIME = {INFERENCE_STRIDE_RUNTIME}')
print(f'ALERT_THRESHOLD          = {ALERT_THRESHOLD}')

In [ ]:
# --- Load fine-tuned model ---
CONFIG_PATH = PROJECT_ROOT / 'config' / 'train_config.yaml'
CKPT_PATH   = PROJECT_ROOT / 'checkpoints' / 'finetune' / 'best_finetune.pt'

config = load_config(CONFIG_PATH)

model = PatchTST(config)
d_in = (
    config['data']['n_patches']
    * config['model']['d_model']
    * config['data']['n_channels']
)  # 73 * 128 * 2 = 18688
model.replace_head(ClassificationHead(d_in=d_in))

import torch
state = torch.load(CKPT_PATH, map_location='cpu', weights_only=True)
model.load_state_dict(state, strict=False)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded from {CKPT_PATH}')
print(f'Total parameters: {n_params:,}')

In [ ]:
# --- Pick one recording (first from train.csv) ---
TRAIN_CSV       = PROJECT_ROOT / 'data' / 'splits' / 'train.csv'
PROCESSED_DIR   = PROJECT_ROOT / 'data' / 'processed' / 'ctu_uhb'

df = pd.read_csv(TRAIN_CSV, dtype={'id': str, 'target': int})
demo_row = df.iloc[0]
REC_ID    = demo_row['id']
REC_LABEL = int(demo_row['target'])
NPY_PATH  = PROCESSED_DIR / f'{REC_ID}.npy'

print(f'Recording : {REC_ID}')
print(f'Label     : {REC_LABEL}  (1=acidemia, 0=normal)')
print(f'NPY path  : {NPY_PATH}')
assert NPY_PATH.exists(), f'Missing: {NPY_PATH}'

signal = np.load(NPY_PATH, mmap_mode='r')  # (2, T)
print(f'Signal shape: {signal.shape}  (2=FHR+UC, T=samples)')
print(f'Duration    : {signal.shape[1] / 4 / 60:.1f} min @ 4 Hz')

In [ ]:
# --- Stage 1: Sliding-window inference (RUNTIME stride for demo speed) ---
# NOTE: Official evaluation always uses INFERENCE_STRIDE_REPRO=1 (Stage 7)
DEMO_STRIDE = INFERENCE_STRIDE_RUNTIME  # 60 samples = 15s steps (demo only)

print(f'Running inference with stride={DEMO_STRIDE} (runtime/demo mode)...')
scores = inference_recording(model, signal, stride=DEMO_STRIDE, device='cpu')

print(f'Total windows scored: {len(scores)}')
starts  = [s for s, _ in scores]
probs   = [p for _, p in scores]
print(f'Score range: [{min(probs):.4f}, {max(probs):.4f}]')

# V5.1 — validate return type
assert all(isinstance(s, int)   for s, _ in scores), 'start_sample must be int'
assert all(0.0 <= p <= 1.0      for _, p in scores), 'score must be in [0, 1]'
print('[V5.1] PASS: inference_recording() returns list of (int, float), score in [0,1]')

In [ ]:
# --- Stage 2a: Extract alert segments ---
segments = extract_alert_segments(scores, threshold=ALERT_THRESHOLD)

print(f'Alert segments found: {len(segments)}')
for i, (start_s, end_s, seg_scores) in enumerate(segments):
    dur_sec = len(seg_scores) * DEMO_STRIDE / 4.0
    print(f'  Segment {i+1}: start={start_s}, end={end_s}, '
          f'n_windows={len(seg_scores)}, max_score={max(seg_scores):.4f}, '
          f'duration~{dur_sec:.0f}s')

# V5.2 — all segments have scores > threshold
for start_s, end_s, seg_scores in segments:
    assert all(s > ALERT_THRESHOLD for s in seg_scores), (
        f'Segment score below threshold {ALERT_THRESHOLD}'
    )
print(f'[V5.2] PASS: extract_alert_segments() returns segments with threshold={ALERT_THRESHOLD}')

In [ ]:
# --- Stage 2b: Compute alert features ---
if segments:
    longest = max(segments, key=lambda seg: len(seg[2]))
    _, _, seg_scores = longest
    feats = compute_alert_features(seg_scores, inference_stride=DEMO_STRIDE, fs=4.0)
else:
    feats = dict(ZERO_FEATURES)
    print('[demo] No alert segments -> using ZERO_FEATURES (assumption S10)')

print('Alert features (4 keys):')
for k, v in feats.items():
    print(f'  {k}: {v:.6f}')

# V5.3 — exactly 4 keys
assert len(feats) == 4, f'Expected 4 features, got {len(feats)}'
expected_keys = {'segment_length', 'max_prediction', 'cumulative_sum', 'weighted_integral'}
assert set(feats.keys()) == expected_keys, f'Wrong keys: {set(feats.keys())}'
print('[V5.3] PASS: compute_alert_features() returns dict with exactly 4 keys')

In [ ]:
# --- V5.4: LR trained on Train only (AGW-26 fix: regex I/O check, not raw substring) ---
import re
train_lr_src = (PROJECT_ROOT / 'src' / 'train' / 'train_lr.py').read_text(encoding='utf-8')

# Check for actual I/O operations on val/test data — docstrings may mention the
# file names as documentation without constituting a data access violation.
# Pattern: read_csv / open / np.load called with val.csv or test.csv as argument
io_violations = re.findall(
    r'(?:read_csv|np\.load|open)\s*\([^)]*(?:test\.csv|val\.csv)',
    train_lr_src,
)
assert len(io_violations) == 0, 'VIOLATION: I/O access to val/test: ' + str(io_violations)
# Sanity: train.csv IS referenced
assert 'train.csv' in train_lr_src, 'train.csv not found in train_lr.py'
print('[V5.4] PASS: train_lr.py has no I/O operations on test.csv or val.csv')

# --- V5.6: stride=REPRO constant used ---
assert 'INFERENCE_STRIDE_REPRO' in train_lr_src, 'INFERENCE_STRIDE_REPRO must appear in train_lr.py'
print(f'[V5.6] PASS: INFERENCE_STRIDE_REPRO (={INFERENCE_STRIDE_REPRO}) is the training stride in train_lr.py')

In [ ]:
# --- V5.5: LR checkpoint exists with stride=1 and full train set (AGW-27 guard) ---
LR_CKPT = PROJECT_ROOT / 'checkpoints' / 'alerting' / 'logistic_regression.pkl'
if LR_CKPT.exists():
    import joblib
    from src.train.train_lr import validate_lr_checkpoint
    ok, msg = validate_lr_checkpoint(LR_CKPT,
                                     expected_stride=INFERENCE_STRIDE_REPRO,
                                     min_n_train=397)  # 90% of 441 train recordings
    assert ok, '[V5.5] FAIL: ' + msg
    print('[V5.5] PASS: ' + msg)
else:
    print(f'[V5.5] SKIP: {LR_CKPT} does not exist yet.')
    print('  Generate via: python src/train/train_lr.py --config config/train_config.yaml --device cuda')
    print('  NOTE: pkl was deleted — the dry-run artifact (stride=60, n_train=5) was stale (AGW-27).')

In [ ]:
# --- Visualization: Score timeline + Alert regions ---
FS = 4  # Hz

time_min = [s / FS / 60.0 for s in starts]  # x-axis in minutes

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# FHR signal
fhr_time  = np.arange(signal.shape[1]) / FS / 60.0
axes[0].plot(fhr_time, signal[0], lw=0.7, color='steelblue', label='FHR')
axes[0].set_ylabel('FHR (normalised)')
axes[0].set_title(f'Recording {REC_ID}  |  label={REC_LABEL} (1=acidemia)')
axes[0].legend(loc='upper right')

# NN score timeline
axes[1].plot(time_min, probs, lw=1.0, color='darkorange', label='P(acidemia)')
axes[1].axhline(ALERT_THRESHOLD, color='red', linestyle='--', lw=1.2,
                label=f'threshold={ALERT_THRESHOLD}')

# Shade alert segments
for seg_i, (start_s, end_s, _) in enumerate(segments):
    x0 = start_s / FS / 60.0
    x1 = end_s   / FS / 60.0
    axes[1].axvspan(x0, x1, color='red', alpha=0.25,
                    label='alert segment' if seg_i == 0 else None)

axes[1].set_xlabel('Time (minutes)')
axes[1].set_ylabel('P(acidemia)')
axes[1].set_ylim(-0.05, 1.05)
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()
print(f'Plot shown for recording {REC_ID}  (stride={DEMO_STRIDE}, n_windows={len(scores)})')

In [ ]:
# --- G5.4: Summary dry-run summary ---
print('=== G5.4 — Inference dry-run summary ===')
print(f'  Recording   : {REC_ID} (label={REC_LABEL})')
print(f'  Windows     : {len(scores)} (stride={DEMO_STRIDE})')
print(f'  Alert segs  : {len(segments)}')
print(f'  Features    : {feats}')
print('[G5.4] PASS: scores list produced, segments extracted, features computed')